# 04b. Graph Engineering ②: 실패별 다음 행동, 부분 실패, 승인과 멱등

04a에서는 Worker가 주고받는 계약(Prompt, Context)과 공유 예산(Harness)을 그래프에 구성했습니다. 이 노트북 **04b**에서는 나머지 두 가지를 이어서 다룹니다. 실패했을 때 무엇을 할지 결정하는 흐름(Graph와 Loop)과, 환불 같은 쓰기 작업을 언제 실행할지 통제하는 절차(승인과 멱등)입니다. 관통 사례는 계속해서 고객 요청 T-1001("결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요.")을 사용합니다.

## 학습 목표

- 실패 종류에 따라 다음 행동을 다르게 정하고, `RetryPolicy`와 `next_action`이 맡는 범위가 서로 다름을 확인합니다.
- 병렬 Worker의 부분 실패를 합류 노드에서 명확히 드러내고, 이미 완료된 결과를 보존한 채 실패한 작업만 다시 실행합니다.
- 사람의 승인을 실제 실행 앞에 배치하고, 실행 재개와 재시도 과정에서 중복 실행이 어떻게 발생하고 어떻게 방지되는지 직접 확인합니다.
- 앞 절에서 만든 부품들을 하나의 그래프에 모으고, 코드·모델·사람이 맡는 노드를 명확히 구분합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 5 | Graph와 Loop: 실패별 다음 행동 | 8 |
| 6 | 부분 실패와 합류 | 5 |
| 7 | 승인과 멱등 | 9 |
| 8 | 완성형 조립 | 5 |
| 9 | 직접 해보기와 정리 | 3 |

## 호출 규모

이 노트북은 `Day03_new/.env`에 설정된 키로 실제 Claude 모델을 호출합니다. 키 없이 실행되는 경로는 없습니다. 모델을 호출하는 셀은 실행 직전에 `announce`가 호출 규모를 한 줄로 안내하며, 비용은 수강생 본인의 키로 청구됩니다. 실패 분류, 합류, 게이트, 멱등처럼 모델의 판단이 필요 없는 부분은 일반 함수로 실행하고 출력에 `"모델 호출 0회"`를 표시합니다. 모델이 할 일이 없는 곳에서 불필요하게 모델을 호출하지 않기 위해서입니다.

| 구분 | 유료 셀 수 | 모델 호출(예고 상한) |
|---|---:|---:|
| 5절 시간 초과 1회 주입 뒤 Worker 복구 | 1 | 12 |
| 7절 모델이 T-1001 환불 제안을 작성 | 1 | 5 |
| 8절 완성형 조립으로 T-1001 처리 | 1 | 16 |
| 기본 실행 합계 | 3 | 예고 상한 33 |

`직접 해보기`의 두 칸은 모델을 호출하지 않습니다. 실제 모델 호출 수는 실행할 때마다 달라질 수 있습니다. 모델이 도구를 몇 번 사용할지는 모델 스스로 결정하기 때문입니다.

아래 코드는 실습에 필요한 패키지와 모듈을 불러오고 API 키, 모델, 관통 사례(T-1001), 평가용 정답표를 준비합니다.

In [ ]:
import inspect
import json
from collections import Counter
from pathlib import Path
from typing import Annotated, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, Send, interrupt

from mas_lab.contracts import (FailureKind, FinalAnswer, Proposal, WorkerResult, WorkerTask, classify_failure, join_results,
                               merge_results, next_action)
from mas_lab.desk import Desk
from mas_lab.evaluation import evaluate_row, load_gold
from mas_lab.hybrid import ROLE_GOALS, ROLES, build_hybrid
from mas_lab.ledger import GateDecision, PaymentLedger, ResponseLost, check_refund, execute_approved
from mas_lab.runner import make_desk, run_case
from mas_lab.structures import scope_line
from mas_lab.trace import ModelMeter, Trace, timeline
from mas_lab.view import show_graph, show_table, write_design_log
from mas_lab.workers import make_worker, run_worker
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

# API 키가 설정되어 있는지 확인합니다. 키가 없으면 한국어 안내와 함께 여기서 멈춥니다.
require_key()
print(key_status())

# 모델 객체를 생성합니다(객체 생성만으로는 API가 호출되지 않습니다).
model = make_model()

# 사례 목록에서 관통 사례(T-1001)와 사후 자동 검사용 정답표를 불러옵니다.
cases = [json.loads(line) for line in Path("data/cases.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
case = next(c for c in cases if c["id"] == "complex-double-pay")
gold = load_gold()  # 사후 자동 검사에만 사용하며, 도구·프롬프트·게이트는 이 값을 참조하지 않습니다.
print("관통 사례:", case["id"], "|", case["question"])
print("측정 장치:", Desk.__name__, Trace.__name__, ModelMeter.__name__)

# 7절의 멱등 재현은 결제 목록과 기준일만 읽으며, 모델이나 도구를 호출하지 않습니다.
case_desk = make_desk(case, label="ledger")

## 04a에서 이어 쓰는 값

5~8절은 04a의 실행 중 메모리에 남아 있는 값을 그대로 이어 쓰지 않습니다. 이 노트북은 04a의 커널 상태 없이도 단독으로 실행되어야 하므로, 위 첫 번째 코드 셀에서 04a와 동일한 준비 과정(키 확인, 모델 생성, T-1001 사례 및 정답표 로드)을 다시 수행했습니다. 이 노트북에서 불러 쓰는 04a의 부품은 5절의 `next_action`, 6절의 `join_results`, 8절의 `build_hybrid` 같은 함수와 클래스들이며, 계약과 예산의 구체적인 의미는 04a의 1~4절에서 다룬 내용을 그대로 따릅니다.

## 5. Graph와 Loop: 실패했다고 무조건 다시 호출하지 않습니다

**문제.** 가장 흔히 볼 수 있는 복구 코드는 `try`로 예외를 잡은 뒤 무조건 다시 호출하는 방식입니다. 일시적인 시간 초과라면 이 방법이 맞을 수 있습니다. 하지만 근거가 부족한 답변을 같은 조건으로 다시 호출하면 비슷한 답변만 돌아오고 비용만 추가로 소모됩니다. 정책상 환불이 불가능한 요청을 다시 호출하면 동일한 거절이 반복되거나, 더 나쁜 경우 정책을 우회하려는 시도로 이어질 수 있습니다. 또한 이미 예산을 초과했는데 다시 호출하면 예산 통제 자체가 의미를 잃게 됩니다.

따라서 실패의 **종류**에 따라 다음 행동을 다르게 정해야 하며, 이 분류와 결정은 모델이 아니라 코드가 맡습니다.

**먼저 예측하세요.** 다음 표의 `다음 행동` 열을 가리고 근거 부족(`insufficient`), 정책 불가(`policy_denied`), 시스템 오류(`system_error`), 예산 초과(`budget_exceeded`) 각각에 대해 어떤 다음 행동이 적절할지 먼저 적어 보세요. 재시도 기회가 남아 있을 때와 모두 소진했을 때를 나누어 생각해 보세요.

아래 코드에서는 `classify_failure`와 `next_action`의 구현 소스를 확인하고, 실패 종류와 조건 조합별 다음 행동을 표로 출력합니다.

In [ ]:
# 실패를 분류하는 함수와 다음 행동을 결정하는 함수의 소스를 출력합니다.
print(inspect.getsource(classify_failure))
print(inspect.getsource(next_action))

# 실패 종류와 재시도·예산 조건 조합에 따른 다음 행동을 표로 확인합니다.
show_table([{"실패 종류": kind.value, "재시도 남음": retries, "예산 남음": left, "다음 행동": next_action(kind, retries_left=retries, budget_left=left)}
            for kind in FailureKind for retries, left in ((1, True), (0, True), (1, False))])
print("모델 호출 0회")

### 그림으로 보기: 실패 종류에 따른 다음 행동

위 표에서 `next_action`을 직접 호출해 확인한 결과를 한 장의 그림으로 정리했습니다. 각 칸은 `next_action`이 선택하는 다음 행동이며, 강조 표시된 칸만 다시 실행(재조사 또는 재시도)하는 행동입니다.

![실패 종류 다섯 가지와 세 조건(재시도 남음·예산 남음, 재시도 0, 예산 없음)에 따른 next_action의 값. 다시 부르는 칸은 insufficient의 reinvestigate와 system_error의 retry 두 곳뿐이다](./assets/04b-1-failure-actions.png)

- 다시 실행하는 칸은 두 곳뿐입니다. 근거가 부족하면 다시 조사(`reinvestigate`)하고, 시스템 오류이면 같은 작업을 다시 시도(`retry`)합니다.
- 정책상 불가와 예산 초과는 재시도 기회가 남아 있어도 다시 호출하지 않습니다. 같은 거절을 반복하거나 예산 한도의 의미를 무너뜨리기 때문입니다.
- 재시도 기회를 모두 소진했거나 예산이 남아 있지 않으면 시스템 오류는 사람에게 넘기고(`escalate`), 근거 부족은 사유를 남긴 채 종료(`stop_with_reason`)합니다.

아래 코드는 실패 분류와 `next_action` 결정에 따라 재시도 여부를 분기하는 상태 그래프(`build_failure_graph`)를 정의합니다.

In [ ]:
class FailState(TypedDict, total=False):
    step: int
    retries_left: int
    kind: str
    action: str
    path: list[str]

def build_failure_graph(act, decide=next_action, retries: int = 1):
    # act(step)가 (오류, 최종 답, 문제 목록)을 반환하며, 실패 분류와 다음 행동은 코드가 결정합니다.
    def attempt(state: FailState) -> dict:
        error, final, problems = act(state.get("step", 0))
        kind = classify_failure(error, final, problems)
        action = decide(kind, retries_left=state["retries_left"], budget_left=not isinstance(error, BudgetExceeded))
        return {"step": state.get("step", 0) + 1, "kind": kind.value, "action": action,
                "retries_left": state["retries_left"] - (action in {"retry", "reinvestigate"}),
                "path": [*state.get("path", []), f"{kind.value} → {action}"]}
    graph = StateGraph(FailState)
    graph.add_node("attempt", attempt)
    graph.add_edge(START, "attempt")
    graph.add_conditional_edges("attempt", lambda s: "attempt" if s["action"] in {"retry", "reinvestigate"} else END, ["attempt", END])
    return graph.compile(), {"retries_left": retries}

print("상태 칸:", list(FailState.__annotations__), "| 모델 호출 0회")

In [ ]:
# 네 가지 실패 시나리오(근거 부족, 정책 불가, 시스템 오류, 예산 초과)를 주입하여 그래프 경로를 확인합니다.
INJECT = {"pass": (None, FinalAnswer(outcome="answered", text="(주입)", evidence_refs=["F-1"]), []),
          "insufficient": (None, FinalAnswer(outcome="insufficient", text="(주입)"), []),
          "denied": (None, FinalAnswer(outcome="denied", text="(주입)"), []),
          "timeout": (TimeoutError("주입한 시간 초과"), None, []),
          "budget": (BudgetExceeded("주입한 예산 초과"), None, [])}
SCENARIOS = {"근거 부족": ["insufficient", "pass"], "정책 불가": ["denied", "pass"], "시스템 오류": ["timeout", "pass"], "예산 초과": ["budget", "pass"]}

def run_scenarios(decide=next_action, retries: int = 1) -> list[dict]:
    rows = []
    for name, script in SCENARIOS.items():
        app, start = build_failure_graph(lambda step, s=script: INJECT[s[min(step, len(s) - 1)]], decide, retries)
        end = app.invoke(start)
        rows.append({"주입": name, "시도 수": end["step"], "경로": " | ".join(end["path"]), "끝난 행동": end["action"]})
    return rows

show_table(run_scenarios())
print("모델 호출 0회")

In [ ]:
# LangGraph의 RetryPolicy가 예외 발생 시점과 정상 반환 시점에 각각 어떻게 동작하는지 비교합니다.
calls = Counter()

def run_policy_graph(node, retry_on=TimeoutError, max_attempts: int = 3) -> str:
    graph = StateGraph(FailState)
    graph.add_node("work", node, retry_policy=RetryPolicy(retry_on=retry_on, max_attempts=max_attempts, initial_interval=0.01, jitter=False))
    graph.add_edge(START, "work")
    graph.add_edge("work", END)
    try:
        return f"끝남: {graph.compile().invoke({'step': 0}).get('kind')}"
    except Exception as error:
        return f"{type(error).__name__}: {str(error)[:50]}"

def recovers(state):
    calls["시간 초과 두 번 뒤 회복"] += 1
    if calls["시간 초과 두 번 뒤 회복"] < 3:
        raise TimeoutError("주입한 시간 초과")
    return {"kind": "pass"}

def always_times_out(state):
    calls["계속 시간 초과"] += 1
    raise TimeoutError("주입한 시간 초과")

def insufficient_answer(state):
    calls["근거 부족을 정상 값으로 반환"] += 1
    return {"kind": "insufficient"}

outcomes = {"시간 초과 두 번 뒤 회복": run_policy_graph(recovers), "계속 시간 초과": run_policy_graph(always_times_out),
            "근거 부족을 정상 값으로 반환": run_policy_graph(insufficient_answer)}
show_table([{"실험": name, "노드 실행 횟수": calls[name], "결과": outcomes[name]} for name in outcomes])
print("모델 호출 0회")

In [ ]:
# 동기 노드에 Send(timeout=...)을 지정했을 때 어떤 예외가 발생하는지 확인합니다.
send_graph = StateGraph(FailState)
send_graph.add_node("work", lambda state: {"kind": "pass"})
send_graph.add_conditional_edges(START, lambda state: [Send("work", {"step": 0}, timeout=0.2)], ["work"])
send_graph.add_edge("work", END)
try:
    print(send_graph.compile().invoke({"step": 0}))
except ValueError as error:
    print("Send(timeout=)을 동기 노드에 적용했을 때:", type(error).__name__, "-", error)
print("모델 호출 0회")

`RetryPolicy`와 `next_action`은 서로 경쟁하는 기능이 아니라 담당하는 범위가 다릅니다. `RetryPolicy`는 노드 실행 중 **예외가 발생했을 때 같은 노드**를 다시 실행합니다. 위 표에서 근거 부족(`insufficient`)을 정상 반환값으로 돌려준 노드가 단 한 번만 실행된 이유도 예외가 발생하지 않았기 때문입니다. 반면 재조사, 미실행 종료, 사람에게 넘기기(`escalate`)처럼 **다른 노드로 경로를 바꾸는 일**은 그래프의 조건부 간선이 맡아야 합니다.

또한 `Send(timeout=)`은 동기 노드에 사용할 경우 `ValueError`가 발생합니다. 동기 Python 코드는 실행 중인 프로세스 안에서 안전하게 강제 취소할 수 없기 때문입니다. 이 과정의 노드는 동기 함수로 작성되어 있으므로 전체 시간 제한은 `Budget`이 담당합니다. 다만 `Budget`의 시간 한도는 다음 예산 소모 시점에 검사하는 방식이라 이미 실행 중인 호출을 도중에 끊지는 않으며, 개별 모델 호출의 제한 시간은 `make_model`의 `timeout` 설정이 담당합니다.

이제 실제 Worker를 사용해 같은 복구 흐름을 실행해 봅니다. 합류 노드가 `join_results`로 작업 상태를 읽고, `next_action`이 `retry`를 반환할 때만 실패한 작업을 새 키(`outage#2`)로 다시 보냅니다. 여기서는 `lookup_availability` 도구에 1회 시간 초과(`timeout_once`)를 주입하며, 복구 경로에만 집중하기 위해 작성자(Writer) 노드 없이 Worker 합류까지만 실행합니다.

아래 코드는 시간 초과 1회 주입 후 실패한 Worker만 다시 실행하여 복구하는 그래프를 구동합니다.

In [ ]:
# lookup_availability에 1회 시간 초과를 주입한 뒤 실패한 Worker만 재시도하여 복구합니다.
announce("시간 초과 1회 주입 뒤 Worker 복구", model_calls=12)
fault_desk, fault_meter = make_desk(case, faults={"lookup_availability": "timeout_once"}, label="fault"), ModelMeter()

class RecoverState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    task: dict

def recover_worker(state: RecoverState) -> dict:
    task = WorkerTask.model_validate(state["task"])
    return {"results": {task.task_id: run_worker(task, fault_desk, model, config={"callbacks": [fault_meter]}).model_dump()}}

def after_recover_join(state: RecoverState):
    results = {k: WorkerResult.model_validate(v) for k, v in state["results"].items()}
    info = join_results(results, list(ROLES))
    retries_left = 0 if any("#" in k for k in results) else 1
    action = next_action(FailureKind.SYSTEM_ERROR, retries_left=retries_left, budget_left=True) if info["failed"] else "respond"
    fault_desk.trace.event("join", "code", failed=info["failed"], action=action)
    if action != "retry":
        return END
    return [Send("worker", {"task": WorkerTask(task_id=f"{t}#2", attempt=2, goal=f"{ROLE_GOALS[t]}\n고객 요청: {case['question']}").model_dump()}) for t in info["failed"]]

recover_graph = StateGraph(RecoverState)
recover_graph.add_node("worker", recover_worker)
recover_graph.add_node("join", lambda state: {"results": {}})
recover_graph.add_conditional_edges(START, lambda state: [Send("worker", {"task": WorkerTask(task_id=r, goal=f"{ROLE_GOALS[r]}\n고객 요청: {case['question']}").model_dump()}) for r in ROLES], ["worker"])
recover_graph.add_edge("worker", "join")
recover_graph.add_conditional_edges("join", after_recover_join, ["worker", END])
recovered = recover_graph.compile().invoke({"results": {}}, config={"recursion_limit": 20})
print("모델 호출:", fault_meter.totals()["model_calls"], "| 결과 키:", sorted(recovered["results"]))

In [ ]:
# 복구 실행 동안 기록된 이벤트, 도구 및 Worker 실행 횟수, 타임라인과 최종 합류 상태를 확인합니다.
fields = ("at", "kind", "actor", "fault", "failed", "action", "month", "invoice_id", "query")
show_table([{k: e.get(k, "") for k in fields} for e in fault_desk.trace.events])
tool_counts = Counter(e["name"] for e in fault_desk.budget.report()["events"] if e["event"] == "tool")
worker_runs = Counter(s["actor"] for s in fault_desk.trace.spans)
show_table([{"구분": "도구", "이름": name, "횟수": count} for name, count in sorted(tool_counts.items())] +
           [{"구분": "Worker 구간", "이름": name, "횟수": count} for name, count in sorted(worker_runs.items())])
print(timeline(fault_desk.trace.spans))
show_table(join_results({k: WorkerResult.model_validate(v) for k, v in recovered["results"].items()}, list(ROLES))["rows"])

In [ ]:
# 시간 초과가 계속 발생하는 경우(timeout_always) 재시도 후 escalate로 종료되는지 확인합니다.
always_desk = make_desk(case, faults={"lookup_availability": "timeout_always"}, label="always")
availability = always_desk.tools("outage")[0]

def call_availability(step: int) -> tuple:
    # 모델 없이 실제 도구를 직접 호출하여 제어 경로만 확인합니다.
    try:
        availability.invoke({"month": "2026-08"})
    except TimeoutError as error:
        return error, None, []
    return None, FinalAnswer(outcome="answered", text="(조회됨)", evidence_refs=["F-AV-2026-08"]), []

always_app, always_start = build_failure_graph(call_availability)
always_end = always_app.invoke(always_start)
show_table([{"주입": "timeout_always", "시도 수": always_end["step"], "경로": " | ".join(always_end["path"]), "끝난 행동": always_end["action"]}])
print("주입 기록:", Counter(e["kind"] for e in always_desk.trace.events), "| 모델 호출 0회")

**관찰할 것.** 복구에 든 비용과 완료 결과 보존 효과를 함께 비교해 보세요. 이벤트 표에서 `fault_injected` 다음에 기록된 `join`의 `action`이 무엇인지 확인합니다. `Worker 구간` 표에 `outage`와 `outage#2`가 각각 기록되어 있다면 재시도가 정상적으로 실행된 것입니다. 이때 `billing`과 `policy`가 각각 1회씩만 실행되었다면 이미 완료된 Worker는 다시 실행되지 않고 보존되었음을 뜻합니다. 즉 재시도 비용은 `outage#2` 구간과 그에 따른 모델 호출만큼만 발생하며, 다른 Worker의 호출 비용은 다시 쓰지 않습니다. 만약 `outage#2`가 보이지 않는다면 모델이 해당 도구를 호출하지 않았거나 다른 경로로 마친 것이므로 `lookup_availability` 호출 횟수와 맨 아래 상태 표를 대조해 확인해 보세요.

두 표의 호출 횟수가 서로 다를 수 있다는 점도 눈여겨보세요. `도구` 행은 호출이 시작될 때 집계하므로 오류 주입으로 실패한 호출까지 포함하지만, 이벤트 표의 `tool` 행은 오류 주입을 통과해 완료된 호출만 기록합니다. 또한 모델이 한 메시지에서 도구를 두 개 동시에 호출했다면 그중 하나만 실패해도 해당 Worker는 `failed` 상태가 되며, 함께 성공한 다른 조회 결과도 Worker 결과로는 반환되지 않습니다. `month` 열에서 모델이 어느 달을 조회했는지도 확인해 보세요(이 인자는 모델이 스스로 결정합니다). 맨 아래 표에서 `outage`의 최종 상태가 `uncertain`으로 나올 수도 있는데, 복구는 실행이 예외 없이 끝났다는 뜻이지 모든 사실을 완벽히 찾았다는 뜻은 아니기 때문입니다.

마지막으로 `timeout_always` 실험 행은 `escalate`로 끝나야 합니다. 재시도 기회를 모두 소진하면 무한정 다시 호출하지 않고 사람에게 넘깁니다. 이는 제어 경로가 설계대로 작동하는지 확인하는 테스트이며 실제 장애율 측정이 아닙니다. 인위적으로 주입한 오류는 실제 운영에서 장애가 얼마나 자주 발생하는지 말해 주지 않습니다.

## 6. 부분 실패와 합류

**문제.** 병렬로 실행된 Worker 셋 중 하나만 실패했습니다. 그래프 자체는 예외로 멈추지 않고 다음 노드로 넘어갑니다. 이때 합류(Join) 노드는 무엇을 해야 할까요? 03에서는 모든 병렬 Worker가 항상 성공한다고 가정했지만, 실제 운영 환경에서는 일부만 성공하고 일부는 실패하는 **부분 실패**가 훨씬 다루기 어렵습니다. 전체가 실패하면 즉시 드러나지만, 일부만 실패하면 겉보기에 정상 완료처럼 지나갈 수 있기 때문입니다.

**먼저 예측하세요.** `outage` Worker가 실패했는데 합류 노드가 성공(`complete`)한 결과만 추려서 작성 노드로 넘긴다고 가정해 봅시다. 작성 노드가 받는 입력에 실패를 알아차릴 만한 흔적이 남아 있을까요? 그 입력만 보고 작성한 최종 답변은 고객에게 어떻게 전달될까요? 가용률 정보가 빠진 이유를 작성자 노드가 알 수 있을지 예측해 보세요.

이 절의 Worker는 모델을 호출하지 않고 미리 정해 둔 결과를 반환하는 일반 함수로 구성합니다. 합류 노드의 제어 흐름을 살펴보는 것이 목적이므로 모델 호출이 필요하지 않습니다.

아래 코드는 성공한 결과만 모아서 넘기는 조용한 합류(`silent_join`)가 부분 실패를 어떻게 지워 버리는지 재현합니다.

In [ ]:
# 실패 재현: 성공한 결과만 모아 넘기는 합류는 실패를 지웁니다.
SAMPLE = {"billing": {"status": "complete", "facts": ["P-9002 1,200,000원은 P-9001과 같은 청구서의 나중 결제입니다."], "evidence_refs": ["F-P-9002"]},
          "policy": {"status": "complete", "facts": ["99.0% 이상 99.5% 미만 구간의 크레딧은 15%입니다."], "evidence_refs": ["E-sample"]},
          "outage": {"status": "failed", "error": "TimeoutError"},
          "outage#2": {"status": "complete", "facts": ["2026-08 가용률은 99.3입니다."], "evidence_refs": ["F-AV-2026-08"]}}
worker_counts = Counter()

class JoinState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    task: str
    writer_input: dict

def sample_worker(state: JoinState) -> dict:
    task_id = state["task"]
    worker_counts[task_id] += 1
    return {"results": {task_id: WorkerResult(task_id=task_id, **SAMPLE[task_id]).model_dump()}}

def silent_join(state: JoinState) -> dict:
    return {"writer_input": {k: v["facts"] for k, v in state["results"].items() if v["status"] == "complete"}}

silent = StateGraph(JoinState)
silent.add_node("worker", sample_worker)
silent.add_node("join", silent_join)
silent.add_conditional_edges(START, lambda state: [Send("worker", {"task": r}) for r in ROLES], ["worker"])
silent.add_edge("worker", "join")
silent.add_edge("join", END)
silent_end = silent.compile().invoke({"results": {}})
print("작성 노드의 입력:", json.dumps(silent_end["writer_input"], ensure_ascii=False))
print("입력에 outage 언급:", "outage" in json.dumps(silent_end["writer_input"]), "| 예외 발생: 없음 | 모델 호출 0회")

In [ ]:
# join_results를 사용해 완료, 불확실, 실패, 결과 누락 네 가지 상태를 모두 드러내고 다음 행동을 확인합니다.
partial = {"billing": WorkerResult(task_id="billing", **SAMPLE["billing"]),
           "policy": WorkerResult(task_id="policy", status="uncertain", uncertainty=["추가 약정을 찾지 못했습니다."]),
           "outage": WorkerResult(task_id="outage", status="failed", error="TimeoutError")}
required = ["billing", "policy", "outage", "credit"]  # credit은 결과가 아예 도착하지 않은 작업입니다.
info = join_results(partial, required)
answers = {"complete": (None, FinalAnswer(outcome="answered", text="(주입)", evidence_refs=["F-1"]), []),
           "uncertain": (None, FinalAnswer(outcome="insufficient", text="(주입)"), []),
           "failed": (TimeoutError("주입"), None, []), "결과 없음": (None, None, [])}
rows = []
for row in info["rows"]:
    kind = classify_failure(*answers[row["상태"]])
    rows.append({**row, "실패 종류": kind.value, "다음 행동": next_action(kind, retries_left=1, budget_left=True)})
show_table(rows)
print("ready:", info["ready"], "| failed:", info["failed"], "| uncertain:", info["uncertain"], "| missing:", info["missing"], "| 모델 호출 0회")

In [ ]:
# 실패한 작업만 새 키(outage#2)로 다시 실행하는 합류 그래프와 merge_results의 충돌 방지를 확인합니다.
worker_counts.clear()

def visible_join(state: JoinState) -> dict:
    results = {k: WorkerResult.model_validate(v) for k, v in state["results"].items()}
    return {"writer_input": join_results(results, list(ROLES))}

def after_visible_join(state: JoinState):
    failed = state["writer_input"]["failed"]
    return [Send("worker", {"task": f"{t}#2"}) for t in failed] if failed else END

visible = StateGraph(JoinState)
visible.add_node("worker", sample_worker)
visible.add_node("join", visible_join)
visible.add_conditional_edges(START, lambda state: [Send("worker", {"task": r}) for r in ROLES], ["worker"])
visible.add_edge("worker", "join")
visible.add_conditional_edges("join", after_visible_join, ["worker", END])
visible_end = visible.compile().invoke({"results": {}})
show_table([{"작업 키": k, "실행 횟수": n} for k, n in sorted(worker_counts.items())])
show_table(visible_end["writer_input"]["rows"])

# merge_results가 같은 키의 동일 값은 허용하고 서로 다른 값은 예외로 막는지 확인합니다.
first = {"outage": WorkerResult(task_id="outage", **SAMPLE["outage"]).model_dump()}
print("같은 값의 재전달:", sorted(merge_results(first, first)))
try:
    merge_results(first, {"outage": WorkerResult(task_id="outage", **SAMPLE["outage#2"]).model_dump()})
except ValueError as error:
    print("같은 키에 다른 결과:", type(error).__name__, "-", error)
print("모델 호출 0회")

**관찰할 것.** 첫 번째 재현 셀(`silent_join`)의 출력은 겉보기에 정상 완료와 전혀 구별되지 않습니다. 예외도 경고도 발생하지 않으며, 작성 노드의 입력에는 `outage`라는 이름조차 남아 있지 않습니다. 작성자 입장에서는 가용률 정보가 왜 없는지 알 길이 없으므로 보상 안내를 통째로 누락하거나 임의의 값으로 메우게 됩니다. 실패를 조용히 삼키는 합류가 실패 그 자체보다 위험한 이유가 바로 여기에 있습니다. 실패 자체는 발견하면 다시 시도할 수 있지만, 조용한 합류는 다시 시도해야 한다는 사실마저 지워 버립니다.

반면 `join_results` 표에서는 네 가지 상태(`complete`, `uncertain`, `failed`, `결과 없음`)가 각각 별도의 행으로 명확히 드러납니다. `failed`와 `결과 없음`은 시스템 오류로 분류되어 재시도(`retry`)로 이어지고, `uncertain`은 근거 부족으로 분류되어 재조사(`reinvestigate`)로 이어집니다. 이처럼 부분 실패와 누락을 드러내는 책임은 합류 노드에 있습니다. 실행 횟수 표에서 `billing`과 `policy`는 각각 1회만 실행되고 `outage`와 `outage#2`가 따로 기록되어 있는지도 확인해 보세요. 재시도는 새 키(`#2`)로 기록하고 `join_results`는 같은 작업의 가장 마지막 시도를 읽으므로, 이전 실패 기록은 보존하면서 상태 표는 최신 결과로 갱신됩니다. 또한 `merge_results`는 같은 키에 서로 다른 값이 들어오면 조용히 덮어쓰지 않고 `ValueError`를 발생시켜 멈춥니다.

다만 이 구조에도 한계는 있습니다. 합류 시점에 도착해 있는 결과를 읽을 뿐, 동시에 실행 중인 두 시도 가운데 뒤늦게 도착한 응답까지 가려내지는 못합니다. 만약 첫 번째 시도가 시간 초과로 처리된 뒤 뒤늦게 응답이 도착하는 시스템이라면 어떤 정보를 추가로 기록해야 할까요? 시도 번호, 시작 시각, 작업 범위를 어떤 키로 묶어 관리할지 생각해 보세요.

04a의 3절 `검증 문제` 열에서 보았던 잘못된 근거 참조도 같은 맥락에서 다룰 수 있습니다. 현재 `join_results`는 작업 상태만 읽고 `validate_result`는 호출하지 않습니다. 만약 합류 노드에서 근거 검증까지 함께 수행한다면, 검증 실패를 어떤 실패 종류로 분류하고 어떤 다음 행동으로 연결해야 할지 함께 생각해 보세요.

### 그림으로 보기: 합류 노드가 실패를 드러내는 방식

위 두 코드 셀에서 비교한 두 그래프를 나란히 나타냈습니다. 두 그래프 모두 세 Worker 가운데 `outage`가 실패한 동일한 입력을 받습니다.

![위는 silent_join이 complete인 결과만 모아 작성 노드에 넘겨 outage가 입력에서 사라지는 흐름이고, 아래는 join_results가 작업별 상태를 드러내고 failed인 outage만 새 키로 다시 Send해 billing과 policy 결과를 그대로 두는 흐름이다](./assets/04b-2-join-visibility.png)

- 위쪽(`silent_join`)은 예외나 경고 없이 그대로 종료됩니다. 작성 노드의 입력에 `outage`라는 흔적이 전혀 남지 않으므로 작성자는 가용률 정보가 빠진 이유를 알 수 없습니다.
- 아래쪽(`join_results`)은 상태가 `failed`인 작업만 `outage#2`라는 새 키로 다시 보냅니다. 이미 완료된 `billing`과 `policy`의 결과는 다시 실행하지 않고 그대로 유지합니다.

## 7. 승인과 멱등

**문제.** 지금까지 다룬 조회 작업의 실패는 다시 실행하면 그만이었습니다. 하지만 환불 같은 **쓰기 작업(부작용)**은 다릅니다. 1,200,000원을 두 번 환불해 버리면 되돌리기가 매우 어렵습니다. T-1001 사례의 P-9002 결제는 100만 원을 초과하므로 `billing-policy.md` 규정에 따라 재무 담당자의 승인이 반드시 필요합니다. 안전한 처리 흐름은 다음 순서를 따릅니다.

처리 제안 → 정책·권한 검증 → 필요한 승인 → 쓰기 실행 → 결과 확인

여기서 모델은 **제안**만 작성합니다. 실제로 실행할지 여부는 코드 게이트와 사람이 결정합니다. LangGraph의 `interrupt`는 노드 실행을 일시 중지하고 외부로 값을 전달하며, 이후 `Command(resume=...)`를 호출해 실행을 재개합니다. 이때 재개 메커니즘이 정확히 어떻게 동작하는지 이해하는 것이 결제 실행의 안전을 좌우합니다.

**먼저 예측하세요.** 한 노드 안에서 `interrupt` 호출 바로 앞에 원장 환불(`refund`) 호출이 놓여 있습니다. 이후 `Command(resume=...)`로 재개하면 노드의 어느 위치부터 다시 실행될까요? `interrupt` 바로 다음 줄부터 이어서 실행될까요, 아니면 노드의 맨 첫 줄부터 다시 실행될까요? 재개까지 모두 끝난 뒤 원장에는 환불 기록이 총 몇 건 남을지 예상해 보고 아래 셀을 실행해 보세요.

아래 코드는 원장 쓰기 호출을 `interrupt` 앞에 두었을 때 어떤 문제가 발생하는지 재현합니다.

In [ ]:
# 실패 재현: 원장 호출을 interrupt 앞에 둔 노드를 재개합니다.
bad_ledger, bad_entries = PaymentLedger(), Counter()

class BadState(TypedDict, total=False):
    request: str
    answer: dict

def refund_then_ask(state: BadState) -> dict:
    bad_entries["노드 첫 줄"] += 1
    bad_ledger.refund("P-9002", 1_200_000)  # 부작용이 interrupt 앞에 위치하며 멱등 키도 없습니다.
    answer = interrupt({"question": "P-9002 1,200,000원 환불을 승인합니까?"})
    bad_entries["interrupt 뒤"] += 1
    return {"answer": answer}

bad_graph = StateGraph(BadState)
bad_graph.add_node("refund_then_ask", refund_then_ask)
bad_graph.add_edge(START, "refund_then_ask")
bad_graph.add_edge("refund_then_ask", END)
bad_app, bad_config = bad_graph.compile(checkpointer=MemorySaver()), {"configurable": {"thread_id": "bad-1"}}
rows = []
bad_app.invoke({"request": "T-1001"}, config=bad_config)
rows.append({"시점": "승인 대기", "노드 첫 줄": bad_entries["노드 첫 줄"], "interrupt 뒤": bad_entries["interrupt 뒤"], "원장 건수": len(bad_ledger.refunds)})
bad_app.invoke(Command(resume={"approved": True}), config=bad_config)
rows.append({"시점": "승인 재개 후", "노드 첫 줄": bad_entries["노드 첫 줄"], "interrupt 뒤": bad_entries["interrupt 뒤"], "원장 건수": len(bad_ledger.refunds)})
show_table(rows)
show_table(bad_ledger.refunds)
print("승인 전에 이미 환불이 실행되었고, 재개 과정에서 한 번 더 실행되었습니다. | 모델 호출 0회")

In [ ]:
# 제안(propose), 게이트(gate), 승인(approve), 실행(execute), 확인(verify)을 별도 노드로 분리해 정의합니다.
class ApprovalState(TypedDict, total=False):
    question: str
    proposal: dict
    decision: dict
    approved: bool
    execution: dict
    verified: dict
    path: list[str]

def approval_nodes(desk: Desk, propose, request_id: str, lose_response: bool) -> tuple[dict, Counter]:
    entries = Counter()
    def step(state, name):
        entries[name] += 1
        return [*state.get("path", []), name]
    def node_propose(state):  # 이미 제안이 들어 있으면 모델을 호출하지 않습니다.
        if state.get("proposal"):
            return {"path": step(state, "propose(입력 사용)")}
        proposal = propose(state["question"])
        return {"proposal": proposal.model_dump(), "path": step(state, "propose")}
    def gate(state):  # 정책 검사만 수행하며, 멱등 키는 코드가 부여합니다.
        proposal = Proposal.model_validate(state["proposal"])
        proposal = proposal.model_copy(update={"idempotency_key": f"{request_id}:{proposal.payment_id}"})
        decision = check_refund(proposal, desk.payments(), desk.scope.as_of)
        return {"proposal": proposal.model_dump(), "decision": decision.model_dump(), "path": step(state, f"gate:{decision.decision}")}
    def approve(state):  # interrupt 앞에 부작용이 없으므로 재개 시 노드가 처음부터 다시 돌아도 안전합니다.
        path = step(state, "approve")
        answer = interrupt({"proposal": state["proposal"], "reasons": state["decision"]["reasons"]})
        if "amount" in answer:  # 승인자가 금액을 수정하면 다시 게이트 검사를 거칩니다.
            return {"proposal": {**state["proposal"], "amount": answer["amount"]}, "path": path}
        return {"approved": bool(answer.get("approved")), "path": path}
    def execute(state):
        proposal, decision = Proposal.model_validate(state["proposal"]), GateDecision.model_validate(state["decision"])
        try:
            execution = execute_approved(proposal, decision, desk.ledger, approved=bool(state.get("approved")), lose_response=lose_response)
        except ResponseLost as error:
            execution = {"executed": "응답 유실", "error": str(error)}
        return {"execution": execution, "path": step(state, "execute")}
    def verify(state):  # 실행 함수의 반환값이 아니라 외부 원장을 직접 다시 조회해 확인합니다.
        record = desk.ledger.find(state["proposal"]["idempotency_key"])
        return {"verified": {"원장에 있음": record is not None, "record": record}, "path": step(state, "verify")}
    return {"propose": node_propose, "gate": gate, "approve": approve, "execute": execute, "verify": verify}, entries

print("상태 칸:", list(ApprovalState.__annotations__), "| 모델 호출 0회")

In [ ]:
# 승인 그래프를 조립하고 시각화하여 노드 간 연결 구조를 확인합니다.
def after_gate(state):
    return {"allow": "execute", "needs_approval": "approve"}.get(state["decision"]["decision"], END)

def after_approve(state):
    if "approved" not in state:  # 승인자가 금액을 수정한 경우 다시 게이트로 보냅니다.
        return "gate"
    return "execute" if state["approved"] else END

def build_approval(desk: Desk, propose=None, *, request_id: str = "T-1001", lose_response: bool = False, with_verify: bool = True):
    nodes, entries = approval_nodes(desk, propose, request_id, lose_response)
    graph = StateGraph(ApprovalState)
    for name, node in nodes.items():
        if name != "verify" or with_verify:
            graph.add_node(name, node)
    graph.add_edge(START, "propose")
    graph.add_edge("propose", "gate")
    graph.add_conditional_edges("gate", after_gate, ["approve", "execute", END])
    graph.add_conditional_edges("approve", after_approve, ["gate", "execute", END])
    graph.add_edge("execute", "verify" if with_verify else END)
    if with_verify:
        graph.add_edge("verify", END)
    return graph.compile(checkpointer=MemorySaver()), entries

show_graph(build_approval(make_desk(case, label="shape"))[0])
print("모델 호출 0회 (그래프 객체를 조립하는 것은 모델 호출이 아닙니다)")

### 그림으로 보기: 부작용이 격리된 승인 그래프

위 셀의 `show_graph`가 출력한 승인 그래프를 부작용(원장 쓰기)이 일어나는 위치 중심으로 다시 정리했습니다.

![propose 뒤 gate가 검사하고 allow면 execute로, needs_approval이면 approve로 간다. approve는 interrupt로 사람을 기다리고 승인되면 execute로, 금액이 바뀌면 gate로 돌아간다. 원장에 쓰는 곳은 execute뿐이고 verify가 원장을 다시 읽는다](./assets/04b-3-approval-graph.png)

- 실제로 원장에 쓰기를 수행하는 노드는 `execute` 하나뿐이며, `approve` 노드에는 부작용이 전혀 없습니다. 재개할 때 `approve` 노드가 처음부터 다시 실행되더라도 아무런 해가 없는 이유입니다.
- `gate`는 정책 검사만 수행하고 멱등 키를 코드로 부여합니다. 만약 승인자가 금액을 수정하면 곧바로 실행하지 않고 `gate`로 돌아가 다시 정책 검사를 받습니다.
- `verify`는 실행 호출의 응답만 믿지 않고 외부 원장을 다시 조회합니다. 네트워크 문제로 응답이 유실되더라도 실제로 돈이 나갔는지 정확히 확인할 수 있습니다.

아래 코드에서는 모델이 T-1001 사례의 환불 제안을 작성하고, 게이트 검사를 거쳐 `approve` 노드의 `interrupt` 지점에서 대기하는 과정을 실행합니다.

In [ ]:
# 모델이 T-1001 환불 제안을 작성하고, 승인 게이트를 거쳐 interrupt에서 대기하는지 확인합니다.
announce("모델이 T-1001 환불 제안을 작성", model_calls=5)
approval_desk, propose_meter = make_desk(case, label="approval"), ModelMeter()
PROPOSE_PROMPT = (f"환불 제안을 작성합니다. {scope_line(approval_desk)} 같은 청구서에 승인된 결제가 두 건 이상이면 중복 결제입니다. "
                  "먼저 승인된 결제는 유효 결제이고 나중 결제가 환불 대상입니다. 기준일 기준 결제일부터 90일 이내의 중복 결제만 이번 환불 대상입니다. "
                  "결제를 조회한 뒤 한 건을 골라 action=refund, payment_id, 결제 금액 amount, reason을 쓰세요. 직접 실행하지 않습니다.")

def model_propose(question: str) -> Proposal:
    agent = make_worker("billing", approval_desk, model, prompt=PROPOSE_PROMPT, response_format=Proposal)
    state = agent.invoke({"messages": [{"role": "user", "content": question}]}, config={"callbacks": [propose_meter], "recursion_limit": 40})
    return state.get("structured_response") or Proposal(reason="구조화 응답 없음")

approval_app, approval_entries = build_approval(approval_desk, model_propose)
approval_config = {"configurable": {"thread_id": "t-1001"}}
waiting = approval_app.invoke({"question": case["question"]}, config=approval_config)
ledger_before = len(approval_desk.ledger.refunds)
print("모델 호출:", propose_meter.totals()["model_calls"], "| 경로:", waiting["path"])
print("제안:", json.dumps(waiting["proposal"], ensure_ascii=False))
print("게이트:", waiting["decision"])
print("멈춘 곳:", approval_app.get_state(approval_config).next, "| interrupt 값:", [i.value["reasons"] for i in waiting.get("__interrupt__", [])])
print("승인 전 원장 건수:", ledger_before)

In [ ]:
# 대기 중인 그래프에 승인(approved=True)을 전달해 재개하고 원장 기록과 노드별 실행 횟수를 확인합니다.
if approval_app.get_state(approval_config).next:
    approved_end = approval_app.invoke(Command(resume={"approved": True}), config=approval_config)
    ledger_after = len(approval_desk.ledger.refunds)
    print("경로:", approved_end["path"])
    print("실행 결과:", approved_end["execution"])
    print("결과 확인:", approved_end["verified"])
    show_table([{"노드": name, "실행 횟수": count} for name, count in approval_entries.items()])
else:
    ledger_after = len(approval_desk.ledger.refunds)
    print("이번 제안은 승인 단계까지 오지 않았습니다. 게이트 결정:", waiting["decision"])
print("승인 전 원장:", ledger_before, "건 | 승인 후 원장:", ledger_after, "건 | 모델 호출 0회")

In [ ]:
# 고정된 제안 입력을 사용해 거절(approved=False)과 금액 수정(amount=1_000_000) 시의 재개 경로를 비교합니다.
FIXED_PROPOSAL = Proposal(action="refund", payment_id="P-9002", amount=1_200_000, reason="T-1001 고정 입력")

def resume_with(thread: str, answer: dict) -> dict:
    # 새 thread와 새 Desk에서 고정 제안으로 시작하며, 모델을 호출하지 않습니다.
    desk = make_desk(case, label=thread)
    app, _ = build_approval(desk)
    config = {"configurable": {"thread_id": thread}}
    app.invoke({"proposal": FIXED_PROPOSAL.model_dump()}, config=config)
    end = app.invoke(Command(resume=answer), config=config)
    return {"재개 값": str(answer), "경로": " → ".join(end["path"]), "게이트 결정": end["decision"]["decision"],
            "게이트 이유": end["decision"]["reasons"][0], "대기 중": bool(app.get_state(config).next), "원장 건수": len(desk.ledger.refunds)}

show_table([resume_with("rejected", {"approved": False}), resume_with("edited", {"approved": True, "amount": 1_000_000})])
print("금액을 수정한 재개는 곧바로 실행되지 않고 게이트를 다시 거칩니다. | 모델 호출 0회")

In [ ]:
# 실패 재현: 응답이 유실된 뒤 키 없이 재시도하면 두 번 환불되고, 키로 재시도하면 한 번만 처리됩니다.
def retry_after_lost(key: str) -> dict:
    ledger = PaymentLedger()
    proposal = FIXED_PROPOSAL.model_copy(update={"idempotency_key": key})
    decision = check_refund(proposal, case_desk.payments(), case_desk.scope.as_of)
    try:
        execute_approved(proposal, decision, ledger, approved=True, lose_response=True)
    except ResponseLost as error:
        lost = str(error)
    after_loss = len(ledger.refunds)
    retried = execute_approved(proposal, decision, ledger, approved=True)
    return {"멱등 키": key or "(없음)", "유실 메시지": lost, "유실 직후 원장": after_loss, "재시도 후 원장": len(ledger.refunds),
            "재시도 응답 refund_id": retried["record"]["refund_id"], "원장의 refund_id": [r["refund_id"] for r in ledger.refunds]}

lost_rows = [retry_after_lost(""), retry_after_lost("T-1001:P-9002")]
show_table(lost_rows)
print("키 없이 재시도:", lost_rows[0]["재시도 후 원장"], "건 | 키로 재시도:", lost_rows[1]["재시도 후 원장"], "건 | 모델 호출 0회")

In [ ]:
# 응답 유실 시 무조건 재실행하기 전에 ledger.find로 외부 원장을 먼저 조회하는 안전한 복구 절차를 확인합니다.
def recover(proposal: Proposal, decision: GateDecision, ledger: PaymentLedger, first_lost: bool) -> list[dict]:
    # 응답이 유실되면 먼저 외부 원장 결과를 조회하여 기록이 없을 때만 다시 실행합니다.
    steps = []
    try:
        record = execute_approved(proposal, decision, ledger, approved=True, lose_response=first_lost)["record"]
        steps.append({"단계": "실행", "결과": "응답 받음", "refund_id": record["refund_id"]})
    except ResponseLost as error:
        steps.append({"단계": "실행", "결과": f"응답 유실: {error}", "refund_id": ""})
        found = ledger.find(proposal.idempotency_key)
        steps.append({"단계": "ledger.find", "결과": "기록 있음. 재시도하지 않습니다." if found else "기록 없음. 다시 실행합니다.", "refund_id": found["refund_id"] if found else ""})
    steps.append({"단계": "원장 건수", "결과": len(ledger.refunds), "refund_id": ""})
    return steps

keyed = FIXED_PROPOSAL.model_copy(update={"idempotency_key": "T-1001:P-9002"})
keyed_decision = check_refund(keyed, case_desk.payments(), case_desk.scope.as_of)
show_table(recover(keyed, keyed_decision, PaymentLedger(), first_lost=True))
print("모델 호출 0회")

**관찰할 것.** 첫 번째 실패 재현 셀의 표에서 `노드 첫 줄` 열과 `interrupt 뒤` 열의 숫자를 비교해 보세요. `Command(resume=...)`를 통한 재개는 멈췄던 줄에서 이어서 실행되는 것이 아니라 **해당 노드를 맨 처음부터 다시 실행**합니다. 따라서 부작용을 `interrupt` 앞에 두면 승인 대기 시점에 이미 원장에 1건이 기록되고, 재개가 끝나면 총 2건이 됩니다. 두 번 환불된 것도 문제이지만, 사람이 승인하기도 전에 돈이 먼저 나갔다는 점이 무엇보다 심각합니다. 반면 올바르게 분리한 승인 그래프의 `실행 횟수` 표에서는 `approve`가 2회, `execute`가 1회 실행된 것을 볼 수 있습니다. `approve` 노드 역시 재개 시 처음부터 다시 실행되지만, 그 안에 부작용이 없기 때문에 안전합니다.

제안 작성 셀의 결과는 모델의 응답에 따라 달라질 수 있습니다. `gate:needs_approval`로 멈췄다면 `멈춘 곳`이 `approve`이고 승인 전 원장 건수는 0건이어야 합니다. 만약 모델이 다른 결제 건이나 다른 금액을 제안했다면 게이트가 거절하여 승인 단계까지 가지 않고 멈추는데, 이때는 거절 사유 자체가 관찰 결과가 됩니다. 어느 경우든 모델의 제안만으로 원장이 변경되는 경로는 없어야 합니다. 이어서 거절(`approved=False`) 행과 금액 수정(`amount=1_000_000`) 행의 `원장 건수`, 그리고 금액 수정 행의 `게이트 이유`도 확인해 보세요. 승인자가 금액을1,000,000원으로 낮추면 곧바로 실행되지 않고 다시 게이트로 돌아가며, 100만 원 이하이므로 추가 승인 없이 실행(`allow`)됩니다. 물론 이것만으로 승인 보안이 완성되는 것은 아닙니다. 승인한 내용과 실제 실행 인자를 암호학적으로 묶는 승인 바인딩은 Day08에서 이어서 다룹니다.

응답 유실 표에서는 `재시도 후 원장` 열을 눈여겨보세요. 원장은 이미 환불 기록을 남긴 뒤 네트워크 응답만 유실된 상태입니다. 이때 멱등 키 없이 재시도하면 원장은 이전과 같은 요청인지 구분할 수 없어 새로운 환불을 한 건 더 생성하지만, 멱등 키(`T-1001:P-9002`)가 있으면 기존과 동일한 `refund_id`를 그대로 반환합니다. 그다음 셀은 한 걸음 더 안전한 복구 순서를 보여 줍니다. 응답이 오지 않았을 때 곧바로 재실행부터 호출하지 않고 `ledger.find`로 외부 원장의 처리 결과부터 먼저 조회하는 방식입니다.

마지막으로 기억할 점은 체크포인터(`MemorySaver`)는 우리 그래프 내부의 진행 상태만 저장할 뿐이라는 사실입니다. `MemorySaver`가 켜져 있어도 첫 번째 재현에서는 환불이 두 번 실행되었습니다. `idempotency_key`는 호출받는 외부 시스템이 이를 지원하고 강제해야만 의미가 있으며, 그래프를 재개한다고 해서 단일 실행(exactly-once)이 자동으로 보장되지는 않습니다. 이 실습의 `PaymentLedger`는 그 동작을 모사한 학습용 객체이므로, 실제 결제 API를 연동할 때는 해당 API가 동일한 키 처리, 키 보관 기간, 조회 API를 어떻게 제공하는지 반드시 공식 문서로 확인해야 합니다.

## 8. 완성형 조립: Workflow 안에 필요한 Agent만 배치합니다

**문제.** 앞 절에서 하나씩 만든 부품들을 하나의 그래프에 모으면 어떤 모습이 될까요? 전체 업무를 단일 Agent 하나에게 통째로 맡기지도 않고, 모든 판단을 고정된 조건문 코드로 짜지도 않습니다. **코드가 전체 경로와 쓰기 권한을 통제하는 Workflow 골격을 잡고, 유연한 판단이 필요한 노드에서만 모델과 Agent를 배치**합니다. 즉 라우팅, 조사, 초안 작성은 모델이 맡고, 범위 확인, 합류, 검증, 게이트, 승인 대기, 원장 실행은 코드가 맡습니다.

소스를 읽으며 다음 세 가지를 찾아보세요.
1. `gate` 노드에는 왜 부작용이 없을까요?
2. `after_join`은 어떤 조건에서 `retry` 노드로 분기할까요?
3. `execute` 노드는 어떤 함수를 통해서만 원장에 접근할까요?

아래 표는 각 노드를 04a 1절에서 다룬 다섯 가지 층(Prompt, Context, Graph, Loop, Harness)에 대응시켜 정리한 것입니다. 그래프 시각화에서 점선 화살표는 조건부 간선을 나타내는데, 어떤 조건에서 어느 갈래로 이동하는지는 그림만으로는 알 수 없으므로 아래 셀이 출력하는 `build_hybrid` 소스 코드에서 직접 확인해 보세요.

In [ ]:
# 완성형 하이브리드 그래프(build_hybrid)의 소스와 구조도, 노드별 담당 층을 확인합니다.
print(inspect.getsource(build_hybrid))
hybrid_shape = build_hybrid(make_desk(case, label="shape"), model)  # 그래프 구조만 생성하며 모델을 호출하지 않습니다.
show_graph(hybrid_shape)
NODE_LAYERS = {"scope": ("Harness", "코드", "범위 밖 고객을 멈춥니다"), "router": ("Prompt, Graph", "모델", "경로와 조사 범위를 고릅니다"),
               "fast": ("Prompt, Context", "Agent", "단순 질문을 바로 답합니다"), "worker": ("Context", "Agent", "WorkerTask를 받아 WorkerResult를 돌려줍니다"),
               "join": ("Graph", "코드", "결과를 합치고 상태를 읽습니다"), "retry": ("Loop", "Agent", "실패한 작업만 새 키로 다시 돌립니다"),
               "writer": ("Context", "Agent", "결과와 calculate_credit으로 답을 씁니다"), "validate": ("Harness, Loop", "코드", "근거와 숫자를 검사하고 재작성합니다"),
               "gate": ("Harness", "코드", "게이트 판단으로 경로를 나눕니다"), "approve": ("Harness", "사람", "interrupt로 승인을 기다립니다"), "execute": ("Harness", "코드", "gate_answer로만 원장에 닿습니다")}
show_table([{"노드": node, "층": layer, "누가 하는가": actor, "역할": role} for node, (layer, actor, role) in NODE_LAYERS.items()])
print("모델 호출 0회")

In [ ]:
# 완성형 조립(hybrid)으로 관통 사례 T-1001을 실행합니다.
announce("완성형 조립으로 T-1001 처리", model_calls=16)
hybrid_row = run_case("hybrid", case, model=model)  # approval이 None이면 승인 대기 상태로 종료됩니다.
print("status:", hybrid_row["status"], "| outcome:", hybrid_row["outcome"], "| 모델 호출:", hybrid_row["model_calls"], "| 시간:", hybrid_row["elapsed_seconds"], "초")
print(hybrid_row["answer"])
print("불확실성:", hybrid_row.get("uncertainty"))

In [ ]:
# 실행 요약 지표, 타임라인, 사후 자동 검사 결과 및 검증 이벤트를 확인합니다.
show_table([{k: hybrid_row[k] for k in ("structure", "status", "outcome", "control_owner", "final_speaker", "model_calls", "tool_calls", "input_tokens", "output_tokens",
                                        "context_bytes", "max_worker_overlap_seconds", "ledger_refunds", "policy_violations")}])
print(timeline(hybrid_row["spans"]))
hybrid_check = evaluate_row(hybrid_row, gold)
show_table([{k: hybrid_check[k] for k in ("자동 검사", "기대 outcome", "실제 outcome", "누락 사실", "금지 포함", "누락 문서", "원장 건수", "실패 칸")}])
print("제안:", hybrid_row.get("proposal"))
print("인용 문서:", hybrid_row["cited_docs"], "| 도구 호출:", Counter(hybrid_row["tools_called"]))
print("검증 사건:", [e["problems"] for e in hybrid_row["events"] if e["kind"] == "validate"])
print("오류:", hybrid_row["errors"] or "없음", "| usage 누락:", hybrid_row["usage_missing"], "| 이 셀의 모델 호출 0회")

**관찰할 것.** 처음의 업무 목표가 끝까지 충족되었는지 차근차근 추적해 보세요. 고객의 요청은 중복 결제 취소와 보상안 확인 두 가지였습니다. 생성된 `answer`에 환불 대상 결제 번호와 금액, 8월 가용률, 크레딧 비율과 계산된 금액, 신청 기한이 모두 들어 있는지 읽고 `자동 검사` 표와 대조해 봅니다. 만약 `실패 칸`에 항목이 있다면 04a 1절의 층별 진단표로 돌아가 어느 층의 문제인지 짚어 보세요. 필수 사실이 빠졌다면 Context 전달의 문제일 수 있고, 잘못된 금액이 적혀 있다면 계산 도구를 쓰지 않은 Prompt나 Harness 검증의 문제일 수 있습니다. 또한 자동 검사를 통과했더라도 실제 원문과 대조하는 확인은 여전히 중요합니다. 반대로 `실패 칸`이 `필수 사실` 하나뿐이고 `누락 사실`이 `9월20일`로 표시되었다면, 먼저 답변 본문에 `2026-09-20`처럼 다른 날짜 표기가 있는지 찾아보세요. 만약 다른 표기로 정확히 적혀 있다면 이는 에이전트의 실패가 아니라 문자열 포함 여부만 보는 자동 검사의 한계입니다.

`timeline` 출력에 `writer` 구간이 두 줄로 나타난다면 검증 후 재작성이 일어난 것입니다. `검증 사건` 출력의 첫 번째 목록이 비어 있지 않다면 `validate` 노드가 문제를 발견해 작성 노드로 돌려보낸 것이고, 두 번째 목록이 비어 있다면 재작성을 통해 그 문제를 해결한 것입니다. 이 재작성에 사용된 모델 호출도 `model_calls` 합계에 포함됩니다(재작성 없이 한 번에 통과했다면 목록이 하나만 출력됩니다).

실행 여부와 미실행 상태의 일관성까지 설명할 수 있어야 합니다. `outcome`이 `needs_approval`이고 `ledger_refunds`가 `0`이면 실제 환불은 아직 실행되지 않은 상태입니다. 만약 답변 문장이 이미 환불을 완료한 것처럼 읽힌다면 `policy_violations` 열과 함께 살펴보세요. 자동 검사는 `outcome`과 원장 건수를 대조할 뿐 문장의 시제 표현까지 세밀하게 판별하지는 못합니다. 마지막으로 `timeline`에서 병렬 Worker 구간들이 시간축에서 서로 겹쳐 실행되었는지도 확인해 보세요. 이 결과 역시 1회 실행 관찰값이므로, 다음 05에서 여러 구조를 같은 조건으로 비교할 때 다시 활용하게 됩니다.

## 직접 해보기

아래 두 실습은 모두 모델을 호출하지 않고 즉시 실행됩니다. 한 번에 조건을 하나씩만 바꾸어 보면서, 실행하기 전에 어떤 결과가 나올지 먼저 예상한 뒤 결과를 확인해 보세요(계약과 전달물을 바꾸어 보는 실습은 04a에 있습니다).

### 해보기 1

`next_action` 함수를 수정해 정책상 불가(`policy_denied`)인 경우에도 재조사(`reinvestigate`)하도록 바꿔 봅니다. 5절의 시나리오 주입 그래프에서 실행 경로가 어떻게 달라지는지 확인해 보세요. 재조사를 세 번까지 허용하면 어떤 숫자가 늘어나고, 그것이 실제 고객 응대에서 어떤 의미일지 생각해 보세요. 특히 `정책 불가` 행의 `시도 수`가 달라지면서 두 번째 시도에서 통과(`respond`)로 끝나는 것을 볼 수 있는데, 이 통과를 실제로 믿어도 될까요? 주입된 테스트 대본(`SCENARIOS`)이 두 번째에 통과하도록 짜여 있기 때문인지, 아니면 재조사가 실제로 회사 규정을 바꿀 수 있기 때문인지 구분해 설명해 보세요.

In [ ]:
# 정책 불가(POLICY_DENIED)에도 재조사를 허용했을 때 시도 횟수와 경로가 어떻게 바뀌는지 비교합니다.
RETRIES = 3

def reinvestigate_policy(kind: FailureKind, *, retries_left: int, budget_left: bool) -> str:
    if kind is FailureKind.POLICY_DENIED:
        return "reinvestigate" if retries_left > 0 and budget_left else "stop_with_reason"
    return next_action(kind, retries_left=retries_left, budget_left=budget_left)

SCENARIOS["정책 불가(계속 거절)"] = ["denied"]
show_table(run_scenarios(reinvestigate_policy, RETRIES))
show_table(run_scenarios(next_action, RETRIES))
del SCENARIOS["정책 불가(계속 거절)"]
print("모델 호출 0회")

### 해보기 2

승인 그래프에서 원장 재조회 노드(`verify`)를 뺀 경우와 넣은 경우에 각각 응답 유실(`lose_response=True`)을 주입해 비교해 봅니다. 표의 두 행에서 `실행 결과`, `결과 확인`, `원장 건수` 열을 비교해 보세요. `verify` 노드가 없으면 실제로 돈이 나갔는지 그래프 상태만으로 알 수 있을까요? 만약 `verify`가 없는 상태에서 운영자가 `"응답 유실"` 메시지만 보고 실패했다고 판단해 다시 실행을 누르면 어떤 일이 벌어질지 설명해 보세요.

In [ ]:
# verify 노드 유무에 따라 응답 유실 발생 시 원장 반영 여부를 확인할 수 있는지 비교합니다.
WITH_VERIFY = [True, False]
rows = []
for with_verify in WITH_VERIFY:
    desk = make_desk(case, label="exercise-4")
    app, _ = build_approval(desk, lose_response=True, with_verify=with_verify)
    config = {"configurable": {"thread_id": f"lost-{with_verify}"}}
    app.invoke({"proposal": FIXED_PROPOSAL.model_dump()}, config=config)
    end = app.invoke(Command(resume={"approved": True}), config=config)
    rows.append({"verify": with_verify, "경로": " → ".join(end["path"]), "실행 결과": end["execution"].get("executed"),
                 "결과 확인": end.get("verified", {}).get("원장에 있음", "알 수 없음"), "원장 건수": len(desk.ledger.refunds)})
show_table(rows)
print("모델 호출 0회")

In [ ]:
# 아래 ANSWER 문자열을 자신의 업무에 맞게 직접 수정해 보세요. 06 노트북에서 이 설계 기록을 모아 읽습니다.
ANSWER = ("무엇을 보고 무엇을 실행할 수 있는가: (내 업무의 Worker마다 볼 수 있는 자료와 쓸 수 있는 도구, 실행은 누가 하는가)\n"
          "언제 재시도하고 언제 전환하는가: (어떤 실패에 재시도하고, 어떤 실패에 사람에게 넘기거나 멈추는가)\n"
          "예산과 종료: (한 실행의 모델 호출 상한, 그 근거, 넘었을 때 무엇을 남기는가)")
path = write_design_log("Context·권한과 예산·종료: 무엇을 보고 무엇을 실행하며 언제 멈출까?", ANSWER)
print("설계 기록을 저장했습니다:", path.relative_to(Path.cwd().resolve()))

## 정리

이 노트북에서 그래프에 추가한 핵심 엔지니어링 요소를 정리합니다. 04a에서 다룬 Prompt, Context, Harness 정리와 함께 이어서 읽어 보세요.

- **Graph.** 합류 노드가 필수 작업별 완료·실패·불확실·누락 상태를 명시적으로 드러내며, 부분 실패를 조용히 빠뜨리지 않습니다.
- **Loop.** 실패의 종류가 다음 행동을 결정합니다. 재시도(`retry`), 재조사(`reinvestigate`), 사유 기록 후 종료(`stop_with_reason`), 사람에게 넘기기(`escalate`)는 서로 대체할 수 없습니다.
- **승인과 멱등.** 원장 변경 같은 부작용은 `interrupt` 뒤의 별도 실행 노드(`execute`)에 격리합니다. `Command(resume=...)` 재개는 노드를 처음부터 다시 실행하기 때문입니다. 체크포인터는 그래프의 진행 상태만 저장하며, 중복 실행 방지는 외부 시스템의 멱등 키(`idempotency_key`)와 결과 재조회(`verify`)에 달려 있습니다.

**Day08에서 이어지는 항목.** 승인 기록과 실제 실행 인자를 묶는 승인 바인딩(도구, 인자 해시, 만료 시간)과 재검증, 입출력 Guardrail, 권한 및 메모리 방어를 이어서 다룹니다.

## 확인 질문

1. 일시적인 시스템 오류(`system_error`)와 근거 부족(`insufficient`)의 다음 행동은 왜 서로 달라야 하나요?
2. 실패한 Worker의 결과를 조용히 빼고 넘어가는 합류(`silent_join`)가 왜 Worker 실패 자체보다 더 위험한가요?
3. 체크포인터(`MemorySaver`)가 있는데도 왜 `idempotency_key`가 별도로 필요한가요? 7절의 첫 번째 실패 재현(`refund_then_ask`)과 두 번째 응답 유실 재현(`retry_after_lost`)은 원인이 어떻게 다른가요?
4. 이 실습의 `PaymentLedger`가 보여 준 멱등 보장을 실제 외부 결제 API에 그대로 기대할 수 없는 이유는 무엇인가요?

다음 **05a·05b**에서는 이 노트북에서 만든 부품들을 고정한 채 에이전트 구조만 바꾸어 비교합니다. 실패 분류에서 출발해 동일한 조건에서 여러 구조를 측정하고, 에이전트를 나누는 이유와 나누지 않는 이유를 수치로 확인합니다.